<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 지식 증류 · 모델 경량화 · 09. Once-for-All

## Once-for-All: Train One Network and Specialize it for Efficient Deployment

- **원 논문:** [Once-for-All: Train One Network and Specialize it for Efficient Deployment](https://openreview.net/forum?id=HylxE1HKwS)
- **저자 / 발표:** Han Cai et al. · ICLR 2020 (2020)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** PyTorch tensor와 autograd, softmax·cross entropy, 선형대수 기초
- **로컬 학습 데이터:** `data/field_curriculum/compression_bench.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 여러 width를 공유하는 supernet을 progressive shrinking으로 학습하고 latency budget에 맞는 subnet을 선택한다.

**축소하거나 생략한 부분:** 원 논문은 depth·width·kernel·resolution과 실제 여러 하드웨어를 지원한다. 실습은 width와 제공된 합성 latency table만 쓴다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | Figure 1: train once, specialize | Task 1–3 | 공유 weight/subnet 선택 |
| §3.1: elastic kernel/depth/width/resolution | Task 1 | width slicing |
| §3.2: progressive shrinking | Task 2 | 큰 subnet→작은 subnet 순서 |
| §3.3: specialization | Task 3 | validation 선택 뒤 고정 subnet test 평가 |

## 미니 재현
모든 기기마다 새 모델을 훈련하지 않고 하나의 supernet weight를 잘라 여러 subnet으로 사용합니다.

## 포트폴리오 해설: 정확도와 효율의 계약

$$w^*=\arg\min_w\mathcal L_{train}(N(S,w)),\quad
s^*=\arg\max_{s\in S,\,\mathrm{lat}(s)\le b}
\mathrm{Acc}(N(s,w^*))$$

- **기호 정의:** S는 subnet 공간, w는 공유 weight, b는 배포 latency budget입니다.
- **수식의 역할:** 한 supernet을 progressive shrinking으로 학습한 뒤 장치 제약에 맞춰 전문화합니다.
- **구현 이유:** 정확도만 남기면 경량화의 비용 절감 주장을 검증할 수 없으므로,
  핵심 변환과 품질·비용 측정을 독립 함수와 method로 분리했습니다.
- **Task/코드 대응:** Task P의 논문 전용 class와 `forward`, `compute_loss`,
  `training_step`, `fit_portfolio_model`, `evaluate_portfolio_model`이
  Figure 1: train once, specialize의 흐름을 명시합니다.
- **입출력 shape:** 특징 [B, 32] → width별 hidden slice → logit [B, 4]
- **평가:** width별 accuracy, 제공 latency table의 budget 만족 여부
- **원문 대비 한계:** 원 논문은 depth·width·kernel·resolution과 실제 여러 하드웨어를 지원한다. 실습은 width와 제공된 합성
latency table만 쓴다.

경량화 실험에서는 정확도만 보지 않고 parameter·sparsity·bit·latency 중 논문이
주장한 비용을 함께 기록해야 합니다. 아래 코드는 단계별 중간값을 남깁니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import math
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
torch.set_default_device(DEVICE)
print(ACCELERATOR.summary())


def locate(relative):
    for base in (Path.cwd(), *Path.cwd().parents):
        path = base / relative
        if path.exists():
            return path
    raise FileNotFoundError(relative)


dataset_path = locate("data/field_curriculum/compression_bench.npz")
raw = np.load(dataset_path, allow_pickle=False)
features = torch.from_numpy(raw["features"]).float().to(DEVICE)
labels = torch.from_numpy(raw["labels"]).long().to(DEVICE)
teacher_logits = torch.from_numpy(raw["teacher_logits"]).float().to(DEVICE)
teacher_weights = torch.from_numpy(raw["teacher_weights"]).float().to(DEVICE)
calibration = torch.from_numpy(raw["calibration_activations"]).float().to(DEVICE)
dense_weights = torch.from_numpy(raw["dense_weights"]).float().to(DEVICE)
train_idx = torch.from_numpy(raw["train_idx"]).long().to(DEVICE)
test_idx = torch.from_numpy(raw["test_idx"]).long().to(DEVICE)
images = torch.from_numpy(raw["images"]).float().to(DEVICE)
image_labels = torch.from_numpy(raw["image_labels"]).long().to(DEVICE)
image_train_idx = torch.from_numpy(raw["image_train_idx"]).long().to(DEVICE)
image_test_idx = torch.from_numpy(raw["image_test_idx"]).long().to(DEVICE)
token_embeddings = torch.from_numpy(raw["token_embeddings"]).float().to(DEVICE)
teacher_hidden = torch.from_numpy(raw["teacher_hidden"]).float().to(DEVICE)
teacher_attention = torch.from_numpy(raw["teacher_attention"]).float().to(DEVICE)
candidate_configs = torch.from_numpy(raw["candidate_configs"]).float()
assert features.shape == (512, 32) and teacher_logits.shape == (512, 4)
assert images.shape == (320, 1, 16, 16)
assert teacher_hidden.shape == (256, 6, 24)
print(
    dataset_path.name, "features", tuple(features.shape), "images", tuple(images.shape)
)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.1: elastic kernel/depth/width/resolution
- **이 셀의 책임:** Task 1
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** width slicing. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 1: 최대 width=24의 weight를 slicing해 8/12/16/24 subnet을 제공하세요.
class WidthSupernet(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x, width):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §3.2: progressive shrinking
- **이 셀의 책임:** Task 2
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 큰 subnet→작은 subnet 순서. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 2: 큰 width에서 시작해 작은 width를 차례로 포함하는 progressive shrinking을 수행하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** Figure 1: train once, specialize / §3.3: specialization
- **이 셀의 책임:** Task 1–3 / Task 3
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 공유 weight/subnet 선택 / validation 선택 뒤 고정 subnet test 평가. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 3: 제공된 latency table에서 budget 이하 후보 중 정확도가 가장 높은 subnet을 고르세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### Task P1 · 압축 대상과 핵심 모델

tensor shape와 비용 단위를 먼저 확인한 뒤 논문 전용 class의 `forward`와
`compute_loss`에 정확도·표현 보존 계약을 명시합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** Figure 1: train once, specialize / §3.1: elastic kernel/depth/width/resolution / §3.2: progressive shrinking / §3.3: specialization
- **이 셀의 책임:** Task 1–3 / Task 1 / Task 2 / Task 3
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 공유 weight/subnet 선택 / width slicing / 큰 subnet→작은 subnet 순서 / validation 선택 뒤 고정 subnet test 평가. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO P-1: 논문의 핵심 layer/pipeline과 비용 평가를 구조화해 구현하세요.
from dataclasses import dataclass


def width_latency_table(widths: tuple[int, ...]) -> dict[int, float]:
    """Collapse the fixture to the width-only search space implemented below."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


@dataclass(frozen=True)
class PortfolioConfig:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    widths: tuple[int, ...] = (24, 16, 12, 8)
    latency_budget_ms: float = 1.3
    learning_rate: float = 0.025
    steps_per_phase: int = 5


def prepare_batch() -> tuple[Tensor, Tensor]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


class OnceForAllPortfolioModel(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, config: PortfolioConfig):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, inputs: Tensor, width: int) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def compute_loss(
        self, inputs: Tensor, targets: Tensor, widths: tuple[int, ...]
    ) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def training_step(
        self, inputs: Tensor, targets: Tensor, widths: tuple[int, ...]
    ) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### Task P2 · 학습·pruning·distillation update

`fit_portfolio_model`에서 optimizer, mask/EMA/quantization 경계와 update 순서를
드러냅니다. 비용 절감이 학습 경로에 미치는 영향을 함께 읽으세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** Figure 1: train once, specialize / §3.1: elastic kernel/depth/width/resolution / §3.2: progressive shrinking
- **이 셀의 책임:** Task 1–3 / Task 1 / Task 2
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 공유 weight/subnet 선택 / width slicing / 큰 subnet→작은 subnet 순서. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: OnceForAllPortfolioModel) -> list[float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### Task P3 · 품질과 비용을 함께 평가

accuracy 또는 reconstruction 품질과 parameter·sparsity·bit·latency를 별도로
기록합니다. 로컬 지표는 원문 하드웨어 benchmark의 대체값이 아닙니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §3.3: specialization
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** features [B,D] + width scalar → subnet logits [B,K]와 latency scalar
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** validation 선택 뒤 고정 subnet test 평가. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def select_subnet_under_budget(
    model: OnceForAllPortfolioModel,
    budget_ms: float,
    validation_inputs: Tensor,
    validation_targets: Tensor,
) -> dict[str, float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def evaluate_portfolio_model(model: OnceForAllPortfolioModel) -> dict[str, float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.